In [16]:
import pandas as pd
import numpy as np

In [17]:
# Load the raw dataset
df_raw = pd.read_csv('updated_data.csv')

print("Original Shape:", df_raw.shape)

# 1. Delete 'Unnamed: 9' column if it exists (do this first so it doesn't trigger NaN drops unnecessarily)
if 'Unnamed: 9' in df_raw.columns:
    df_raw = df_raw.drop(columns=['Unnamed: 9'])
    print("Dropped 'Unnamed: 9' column.")

# 2. Drop any row that contains any NaN value across the remaining columns
df_raw = df_raw.dropna(how='any')

print("Cleaned Shape:", df_raw.shape)
df_raw.head(2)

Original Shape: (3400, 11)
Dropped 'Unnamed: 9' column.
Cleaned Shape: (3359, 10)


,scheme_name,slug,details,benefits,eligibility,application,documents,level,schemeCategory,tags
0,"""Immediate Relief Assistance"" under ""Welfare a...",ira-wrflsncs,"The scheme ""Immediate Relief Assistance"" is a ...","₹ 1,00,000, in two installments of ₹ 50,000 ea...",The applicant should be the family (legal heir...,Step 1: The interested applicant should visit ...,Photograph of the Family (Legal Heir) of the M...,State,"Agriculture,Rural & Environment, Social welfar...","Missing, Fisherman, Relief, Financial Assistan..."
1,AICTE SHORT TERM TRAINING PROGRAMME-SFURTI SCHEME,astpss,"Short Term Training Programme-SFURTI Program, ...","Financial Assistance : Limit of funding ₹ 4,00...",The institution should be AICTE approved.,Registration of New Institute: Step 01: Visit ...,Feedback Form Copy of Proceedings Completion R...,Central,Education & Learning,"Trainings, Financial Assistance, AICTE"


In [18]:
df_raw.info()

<class 'pandas.DataFrame'>
Index: 3359 entries, 0 to 3399
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   scheme_name     3359 non-null   str  
 1   slug            3359 non-null   str  
 2   details         3359 non-null   str  
 3   benefits        3359 non-null   str  
 4   eligibility     3359 non-null   str  
 5   application     3359 non-null   str  
 6   documents       3359 non-null   str  
 7   level           3359 non-null   str  
 8   schemeCategory  3359 non-null   str  
 9   tags            3359 non-null   str  
dtypes: str(10)
memory usage: 12.3 MB


In [19]:
# The specific columns we want to turn into distinct chunk documents
import pandas as pd
import re
import unicodedata

def clean_for_llm_and_tts(text: str) -> str:
    """
    Cleans text formatting for the LLM/TTS but preserves cases and grammatical punctuation.
    """
    if not text or pd.isna(text): return ""

    # 1. Normalize unicode (removes invisible artifacts like \ufeff)
    text = unicodedata.normalize('NFKD', str(text))
    # 2. Replace all newlines (\n) and carriage returns with a single space
    text = re.sub(r'[\r\n]+', ' ', text)
    # 3. Clean up excessive/repeated double quotes (e.g. ""Immediate"")
    text = re.sub(r'"+', '"', text)
    # 4. Collapse multiple spaces into a single space
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

def clean_for_bm25_search(text: str) -> str:
    """
    Aggressively strips punctuation and lowercases for BM25 sparse keyword matching.
    """
    if not text or pd.isna(text): return ""

    # 1. Lowercase
    text = text.lower()
    # 2. Keep ONLY letters, numbers, and spaces (removes all punctuation)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    # 3. Collapse multiple spaces
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

# The specific columns we want to turn into distinct chunk documents
sections_to_chunk = ['details', 'benefits', 'eligibility', 'application', 'documents']
processed_rows = []

for index, row in df_raw.iterrows():
    # Extract Metadata
    scheme_name = str(row.get('scheme_name', '')).strip()
    scheme_slug = str(row.get('slug', '')).strip()
    gov_level = str(row.get('level', '')).strip()

    # Extract fields for sparse_text
    tags = str(row.get('tags', ''))
    category = str(row.get('schemeCategory', ''))

    # Clean up 'nan' strings that pandas might cast
    tags = tags if tags.lower() != 'nan' else ""
    category = category if category.lower() != 'nan' else ""

    # Construct, clean, and aggressively strip the sparse keyword metadata
    raw_sparse_text = f"{scheme_name} {category} {gov_level} {tags}"
    sparse_text = clean_for_bm25_search(raw_sparse_text)

    # Clean the scheme name for prepending
    clean_scheme_name = clean_for_llm_and_tts(scheme_name)

    for section in sections_to_chunk:
        section_content = row.get(section)

        # Check if the section actually contains valid text
        if pd.notna(section_content) and str(section_content).strip() not in ["", "nan"]:

            # Clean the chunk content preserving grammar and cases
            clean_content = clean_for_llm_and_tts(str(section_content))

            # Prepend the cleaned scheme name
            document_text = f"Scheme Name: {clean_scheme_name} {clean_content}"

            processed_rows.append({
                "document_text": document_text,
                "scheme_id": scheme_slug,
                "scheme_name": clean_scheme_name,
                "government_level": gov_level,
                "sparse_text": sparse_text,
                "section": section
            })

df_processed = pd.DataFrame(processed_rows)
print(f"Reshaped {len(df_raw)} schemes into {len(df_processed)} distinct, perfectly cleaned chunks.")

Reshaped 3359 schemes into 16795 distinct, perfectly cleaned chunks.


In [20]:
# Let's verify how the chunks look, especially the document_text
print("Columns in new dataset:", df_processed.columns.tolist())

# Check the first row's document_text to confirm the Scheme Name is prepended
print("\n--- Sample Document Text ---")
print(df_processed.iloc[0]['document_text'])
print("----------------------------\n")

df_processed.head()

Columns in new dataset: ['document_text', 'scheme_id', 'scheme_name', 'government_level', 'sparse_text', 'section']

--- Sample Document Text ---
Scheme Name: "Immediate Relief Assistance" under "Welfare and Relief for Fishermen During Lean Seasons and Natural Calamities Scheme" The scheme "Immediate Relief Assistance" is a Sub-Component under the scheme "Welfare and Relief for Fishermen During Lean Seasons and Natural Calamities Scheme". The scheme is extended to all the regions of the Union territory of Puducherry. The scheme is introduced with the objective of extending financial assistance to the fishermen's families to compensate for the loss due to the missing breadwinner and to support them financially to run their family.
----------------------------



,document_text,scheme_id,scheme_name,government_level,sparse_text,section
0,"Scheme Name: ""Immediate Relief Assistance"" und...",ira-wrflsncs,"""Immediate Relief Assistance"" under ""Welfare a...",State,immediate relief assistance under welfare and ...,details
1,"Scheme Name: ""Immediate Relief Assistance"" und...",ira-wrflsncs,"""Immediate Relief Assistance"" under ""Welfare a...",State,immediate relief assistance under welfare and ...,benefits
2,"Scheme Name: ""Immediate Relief Assistance"" und...",ira-wrflsncs,"""Immediate Relief Assistance"" under ""Welfare a...",State,immediate relief assistance under welfare and ...,eligibility
3,"Scheme Name: ""Immediate Relief Assistance"" und...",ira-wrflsncs,"""Immediate Relief Assistance"" under ""Welfare a...",State,immediate relief assistance under welfare and ...,application
4,"Scheme Name: ""Immediate Relief Assistance"" und...",ira-wrflsncs,"""Immediate Relief Assistance"" under ""Welfare a...",State,immediate relief assistance under welfare and ...,documents


In [21]:
# Save to the exact filename your CSVLoader script in ingest.py is expecting
output_filename = "schemes_compact_cleaned_merged_chunked.csv"

# We use index=False so we don't write row numbers into the CSV
df_processed.to_csv(output_filename, index=False, encoding='utf-8')

print(f"Successfully saved to {output_filename}!")
print("You can now run your voice/retrieval/ingest.py pipeline.")

Successfully saved to schemes_compact_cleaned_merged_chunked.csv!
You can now run your voice/retrieval/ingest.py pipeline.


In [26]:
df_processed['sparse_text'][6]

'aicte short term training programme sfurti scheme education learning central trainings financial assistance aicte'

In [27]:
df_processed['section'].value_counts()

section
details        3359
benefits       3359
eligibility    3359
application    3359
documents      3359
Name: count, dtype: int64